In [1]:
import sys
sys.path.append('..')
sys.path.append('../mmdetection')
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
import gc
import numpy as np
import pandas as pd
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules
from mmdet.registry import VISUALIZERS
import torch
from tqdm import tqdm
from turtlewatch_egypt import TurtleWatchEgypt_NYED, TurtleWatchEgypt_AED 
from wildlife_datasets.datasets import WildlifeDataset

In [2]:
root_nyed = '/data/wildlife_datasets/turtle_watch/2022 PICS + online database'
root_aed = '/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR'
root_results = '/data/wildlife_datasets/turtle_watch/2022 PICS + online database_crops'
save_threshold = 0.8
min_pixels = 100

TurtleWatchEgypt_NYED.get_data(root_nyed, force=True)
TurtleWatchEgypt_AED.get_data(root_aed, force=True)

DATASET TurtleWatchEgypt_NYED: DOWNLOADING STARTED.
DATASET TurtleWatchEgypt_NYED: EXTRACTING STARTED.
Ignoring corrupted files:
11 November 2022/SATA/2022_11_09 AT/SIGHTING #1 TOMEK LECHOSLAW/TOMEK LECHOSLAW/IMG_3643.JPG
05 May 2022/MMUB/2022_05_16 AM Miriam Tercon/Marsa Mubarak 2022_05_16_12AM_Miriam Tercon/1 17 NO ID/17 NO ID/NOID_L_MM_2022_05_16_MMUB_AM_1_MIRIAM tercon3.JPG
05 May 2022/MMUB/2022_05_16 AM Miriam Tercon/Marsa Mubarak 2022_05_16_12AM_Miriam Tercon/1 17 NO ID/17 NO ID/NOID_L_MM_2022_05_16_MMUB_AM_1_MIRIAM tercon2.JPG
Creating artificial encounters:
05 May 2022/MMUB/2022_05_16 AM Miriam Tercon/Marsa Mubarak 2022_05_16_12AM_Miriam Tercon/1 17 NO ID/17 NO ID
05 May 2022/MMUB/2022_05_16 AM Miriam Tercon/Marsa Mubarak 2022_05_16_12AM_Miriam Tercon/2 18 NO ID/18 NO ID
05 May 2022/MMUB/2022_05_16 AM Miriam Tercon/Marsa Mubarak 2022_05_16_12AM_Miriam Tercon/3 19 NO ID/19 NO ID
06 June 2022/TORF/torfa 2022_06_28
07 July 2022/SHAL/2022_07_24 MT
07 July 2022/TORFA/2022_07_12
08 A

In [3]:
dataset_nyed = TurtleWatchEgypt_NYED(root_nyed)
dataset_aed = TurtleWatchEgypt_AED(root_aed)

In [4]:
identities = np.union1d(dataset_aed.df['identity'], dataset_nyed.df['identity'])
print(len(identities))
identities

312


array(['abdu', 'aero', 'agnes', 'agnese', 'agnieszka', 'alessandra',
       'alice', 'alizè', 'alke', 'altea', 'amaly', 'ambesa', 'amina',
       'amira', 'ammar', 'amun', 'andrzej', 'angelo', 'anna', 'annette',
       'annvd', 'anriz', 'ansofie', 'aquamarine', 'archie', 'ariel',
       'armin', 'arwa', 'asiel', 'asunk3', 'asunk6', 'asunk8', 'aurelia',
       'ava', 'baba dana', 'bacri', 'baju', 'barbara', 'basil', 'bea',
       'beany', 'bella', 'belle', 'bernhard', 'betzi', 'bibi', 'bibo',
       'bigmama', 'bigos', 'blackjack', 'blu', 'boma', 'bonny', 'brave',
       'bromley', 'bruno', 'bubson', 'camille', 'carmencita', 'caroline',
       'carpaccio', 'castella', 'charly', 'chino', 'christian',
       'christoph', 'ciuca', 'claus', 'clorofilla', 'coral', 'crush',
       'cuccy', 'dabb3 ei', 'dabb4 ei', 'dalí', 'daniel', 'daniela',
       'daniëlle', 'dario', 'dasa', 'dazzer', 'dea', 'denise', 'didier',
       'domenico', 'doris', 'eaton', 'eduard', 'egunk9', 'eirene',
       'elega

In [5]:
import re

def modify_str(x):
    if not isinstance(x, str):
        return x
    x = re.sub(r'\d+', '', x)
    for _ in range(10):
        x = x.removesuffix(' - copia')
    x = x.removesuffix(' ()')
    return x

encounters = []
for encounter_id, df_encounter in dataset_nyed.df.groupby('encounter_id'):
    identity = df_encounter['identity']
    if identity.nunique() != 1:
        raise ValueError('Identity is not unique')

    dates = [dataset_nyed.extract_info(i)[3] for i in df_encounter.index]
    dates = pd.Series(dates).dropna().unique()
    if len(dates) == 0:
        dates = df_encounter['date'].dropna().unique()
        dates = pd.to_datetime(dates, format='%Y-%m-%d')
        dates = [str(min(dates))[:10]]
    if len(dates) > 1:
        raise ValueError('Dates not unique')

    places = [dataset_nyed.extract_info(i)[4] for i in df_encounter.index]
    places = pd.Series(places).dropna().unique()
    if len(places) > 1:
        raise ValueError('Places not unique')

    authors = [dataset_nyed.extract_info(i)[6] for i in df_encounter.index]
    authors = [modify_str(x) for x in authors]
    authors = pd.Series(authors).dropna().unique()    
    if len(authors) > 1:
        authors = []

    folder = os.path.commonprefix(df_encounter['path'].to_list())
    folder = folder[:folder.rfind(os.path.sep)+1]

    encounters.append({
        'encounter_id': encounter_id,
        'identity': identity.iloc[0],        
        'n photos': len(df_encounter),
        'date': dates[0] if len(dates) >= 1 else np.nan,
        'place': places[0] if len(places) >= 1 else np.nan,
        'author': authors[0] if len(authors) >= 1 else np.nan,
        'folder': folder,
    })
encounters = pd.DataFrame(encounters)
encounters.to_csv(f'{root_nyed}/encounters.csv', index=False)

In [6]:
dataset_nyed.df.identity.isnull().sum(), (dataset_nyed.df.identity == 'unknown').sum()

(0, 501)

In [7]:
root_models = '/data/wildlife_datasets/models/detection'
model_name = 'htc_r50'

path_config = f'/data/wildlife_datasets/models/detection/{model_name}/{model_name}.py'
path_checkpoint = f'/data/wildlife_datasets/models/detection/{model_name}/epoch_12.pth'

register_all_modules()
model = init_detector(path_config, path_checkpoint, device='cuda')

visualizer = VISUALIZERS.build(model.cfg.visualizer)
visualizer.dataset_meta = model.dataset_meta

Loads checkpoint by local backend from path: /data/wildlife_datasets/models/detection/htc_r50/epoch_12.pth


/home/sadda/Projects/turtlewatch-egypt/venv_detection/lib/python3.10/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


In [8]:
file_name = f'{root_nyed}/segmentation.csv'
if not os.path.exists(file_name):
    image_ids = np.empty(0, dtype=int)
    bboxes = np.empty((0,4))
    scores = np.empty(0, dtype=float)
    labels = np.empty(0, dtype=int)
    for i, (_, df_row) in enumerate(tqdm(dataset_nyed.df.iterrows(), total=len(dataset_nyed.df))):
        with torch.no_grad():
            result = inference_detector(model, f'{root_nyed}/{df_row["path"]}')
        image_ids = np.concatenate((image_ids, len(result.pred_instances['bboxes'])*[df_row['image_id']]))
        bboxes = np.concatenate((bboxes, result.pred_instances['bboxes'].cpu().numpy()))
        scores = np.concatenate((scores, result.pred_instances['scores'].cpu().numpy()))
        labels = np.concatenate((labels, result.pred_instances['labels'].cpu().numpy()))
        if i % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    segmentation = pd.DataFrame({
        'image_id': image_ids,
        'bbox': list(bboxes),
        'score': scores,
        'label': [model.dataset_meta['classes'][x] for x in labels]
    })
    segmentation['image_id'] = segmentation['image_id'].astype(int)
    segmentation.to_csv(file_name, index=False)
else:
    segmentation = pd.read_csv(file_name)

In [9]:
def convert_bbox(x):
    x = x.split('[')[1]
    x = x.split(']')[0]
    x = ' '.join(x.split())
    bbox = [float(y) for y in x.split(' ')]
    return [bbox[0], bbox[1], bbox[2]-bbox[0], bbox[3]-bbox[1]]

segmentation = segmentation[segmentation['score'] >= save_threshold]
segmentation = segmentation[segmentation['label'] == 'flipper']
segmentation = pd.merge(dataset_nyed.df, segmentation, on='image_id')
segmentation['bbox'] = segmentation['bbox'].apply(convert_bbox)
idx = segmentation['bbox'].apply(lambda x: min(x[2:])) >= min_pixels
segmentation = segmentation[idx].reset_index(drop=True)

dataset = WildlifeDataset(root_nyed, df=segmentation, img_load='bbox')

In [10]:
root_images = os.path.join(root_results, 'images')
os.makedirs(root_images, exist_ok=True)

for i in range(len(dataset)):
    dataset[i].save(os.path.join(root_images, f'{i:05d}.png'))

Corrupt JPEG data: 33 extraneous bytes before marker 0xd9
Corrupt JPEG data: 33 extraneous bytes before marker 0xd9
Corrupt JPEG data: premature end of data segment


In [11]:
segmentation['path'] = [f'images/{i:05d}.png' for i in range(len(segmentation))]
segmentation = segmentation.drop(['image_id', 'label', 'bbox'], axis=1)
segmentation.to_csv(os.path.join(root_results, 'metadata.csv'))